# Lekcija 12 - Zmanjšanje zgodovine klepeta z agentovim zapiskom

Ta zvezek prikazuje, kako upravljati s kontekstom v dolgih pogovorih z uporabo Microsoft Agent Framework. Ko se pogovori razvijajo, se število tokenov povečuje — na koncu preseže okno konteksta modela. To rešujemo z **vzorcem povzema konteksta** in **agentovim zapiskom** za trajni spomin.

## Kaj se boste naučili:
1. **Zakaj je upravljanje konteksta pomembno**: Razumevanje omejitev tokenov in oken konteksta
2. **Agenti, ki se zavzemajo za kontekst**: Gradnja agentov, ki upravljajo svoj lasten kontekst pogovora
3. **Vzorec povzema konteksta**: Uporaba orodij za stiskanje zgodovine pogovorov
4. **Agentov zapisek**: Trajni spomin, ki preživi zmanjšanje konteksta

## Predpogoji:
- Nastavitev Azure OpenAI z konfiguriranimi okoljskimi spremenljivkami
- Razumevanje osnovnih konceptov agentov iz prejšnjih lekcij


## Namestitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Zakaj je upravljanje konteksta pomembno

Vsak LLM ima omejeno **kontekstno okno** — največje število tokenov, ki jih lahko obdela v eni zahtevi. Ko se večkrožni pogovor nadaljuje:

- **Število tokenov linearno raste** z vsakim uporabniškim sporočilom in odgovorom asistenta.
- **Tokeni poziva prevladujejo stroške**, ker se celotna zgodovina pošlje znova vsakokrat.
- Na koncu pogovor **preseže kontekstno okno** in model bodisi odsekava podatke bodisi javi napako.

### Strategije za upravljanje konteksta

| Strategija | Kako deluje | Kompromis |
|---|---|---|
| **Odsekavanje** | Odstrani najstarejša sporočila | Izgubi zgodnji kontekst |
| **Povzetek** | Stisne starejša sporočila v povzetek | Nekaj podrobnosti izgubljenih, a ključne točke ostanejo |
| **Zvezek / Zunanja pomnilnik** | Shrani ključne informacije zunaj pogovora | Zahteva klice orodij, a preživi katerokoli zmanjšanje |

V tej zvezku združujemo **povzetek** z orodjem **zvezek**, da lahko agent ohrani kontinuiteto tudi, ko je zgodovina pogovora stisnjena.


## Ustvarjanje agenta, ki zaznava kontekst


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulacija dolgega pogovora

Pojdimo skozi večkratni pogovor, da vidimo, kako se kontekst nabira. Agent naj ohrani ključne podrobnosti (preferenče, proračun, datume potovanja) med posameznimi pogovori in pokaže kontinuiteto.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Opazite, kako agent ohranja kontekst iz prejšnjih pogovorov — ve za Japonsko, sushija, templje, fotografijo, proračun 3000 $, samostojno potovanje in časovni okvir april. V kratkem pogovoru to deluje dobro, vendar pa z rastjo pogovora celotna zgodovina postane draga za ponovno pošiljanje.

Nadaljujmo pogovor z več odzivi, da vidimo nalaganje konteksta:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzorec povzemanja konteksta

Ko se pogovor razvija, lahko uporabimo **orodje za povzemanje**, da zgoščeno oblikujemo nabrani kontekst. Agent pokliče to orodje, da zabeleži ključne preference, tako da tudi če se starejša sporočila izbrišejo, so bistvene informacije ohranjene.

Ta vzorec je osnovni gradnik za bolj sofisticirano zmanjševanje zgodovine:
1. Agent prepozna ključna dejstva iz pogovora
2. Pokliče orodje za povzemanje, da jih shrani
3. Starejša sporočila je varno odstraniti, ker povzetek zajema, kar je pomembno

Spodaj definiramo orodje `summarize_preferences`, ki ga lahko agent pokliče, da zabeleži kratek povzetek tega, kar se je naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Povzetek

V tej lekciji ste se naučili, kako upravljati s kontekstom v dolgotrajnih pogovorih agentov z uporabo Microsoft Agent Frameworka:

### Ključni koncepti
- **Kontekstna okna so omejena** — vsak žeton v zgodovini pogovora stane denar in se šteje v omejitev.
- **Orodja za povzema** omogočajo agentu, da zgošči nabrani kontekst v kompaktne povzetke, kar zmanjšuje porabo žetonov, hkrati pa ohranja bistvene informacije.
- **Zvezki agenta** nudijo trajni zunanji spomin, ki preživi katerokoli zmanjšanje pogovora.

### Kaj ste zgradili
- **Agent, ki je zavedajoč konteksta**, in vzdržuje kontinuiteto v večkrožnih pogovorih
- **Orodje za povzema** (`summarize_preferences`), ki zabeleži ključne uporabniške podatke v kompaktni obliki
- **Večkrožni pogovor**, ki prikazuje ohranjanje konteksta in obvladovanje sprememb

### Uporabe v resničnem svetu
- **Bots za pomoč strankam**: Zapomnijo si preference skozi dolge podporne seje
- **Osebni asistenti**: Spremljajo tekoče projekte brez ponovnega razlaganja konteksta
- **Izobraževalni tutorji**: Ohranjajo napredek študenta skozi veliko interakcij

### Naslednji koraki
- Izvedba orodja za zvezek z vztrajno shrambo na osnovi datotek
- Dodajanje samodejnega krajšanja zgodovine po povzemanju
- Združevanje z vektorskimi podatkovnimi bazami za semantično iskanje spomina
- Izgradnja agentov, ki lahko nadaljujejo pogovore po dneh s polnim kontekstom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo storitve umetne inteligence za prevajanje [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da lahko samodejni prevodi vsebujejo napake ali netočnosti. Avtentičen vir je izvirni dokument v njegovem izvorni jeziku. Za kritične informacije priporočamo strokovni človeški prevod. Ne odgovarjamo za kakršna koli nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
